In [ ]:

import base64, json, os, pathlib, subprocess, sys, time, zipfile, torch
T0 = time.monotonic(); CAP = 14500.0
w = pathlib.Path("/kaggle/working"); p = w / "payload"; p.mkdir(exist_ok=True)
BLOB = {'upstream.zip': '...', 'daniel_arm.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiRFJBRlRfVU5WRVJJRklFRDogcnVuIHRoZSAqcGFwZXIgcmVjaXBlKiBvZiBHb256YWxleiBDYXJyaWxsbydzIGluZGlyZWN0IFBJTk4gKFBJTk5zX3BpZXpvZWxlY3RyaWNpdHkgQCBlNzg1NmYxLCAyMDI2LTEwLTAyKSB3aXRoIGVpdGhlciBpdHMgb3duDQpNTFAgKDN4NTAgU2lMVSwgcGhhc2UtZW5yaWNoZWQgc2hhcmVkIHRydW5rKSBvciBhbiBSQkYtUElLQU4gdHJ1bmsgc3dhcHBlZCBpbiBmb3IgYG1vZGVsLm5ldGAsIGV2ZXJ5dGhpbmcgZWxzZSB1bmNoYW5nZWQNCihsYXllcmVkIHNhbXBsaW5nIHdpdGggMjA0OCBpbnRlcmlvciBwb2ludHMgcGVyIGxheWVyLCBpbnRlcmZhY2UvYm91bmRhcnkgcG9pbnRzLCByZXNhbXBsaW5nIGV2ZXJ5IDIwMCBlcG9jaHMsIHBlci1maWVsZCBzY2FsZXMsDQptZWNoYW5pY2FsIGNvbnN0aXR1dGl2ZSBzdG9wLWdyYWRpZW50IHJvdXRpbmcsIGhhcmQgdm9sdGFnZSBsaWZ0aW5nLCAyMCwwMDAgQWRhbSBlcG9jaHMgYXQgbHIgMWUtMywgZmxvYXQ2NCwgbm8gTC1CRkdTKS4NClNjb3JpbmcgaXMgZG9uZSBoZXJlIGFnYWluc3QgdGhlIEZFTSBncmlkIChwb2xpbmdfc2lnbj0tMSkgc28gTUxQIGFuZCBSQkYgc2hhcmUgb25lIGV2YWx1YXRvci4NCg0KdXNhZ2U6IHB5IC0zIGRhbmllbF9hcm0ucHkgLS1kYW5pZWwtcm9vdCA8cmVwbyBjbG9uZT4gLS1hcm0gbWxwfHJiZiAtLXNlZWQgMjAyNjA3MjggLS1lcG9jaHMgMjAwMDAgLS1vdXQgRElSIC0tZmVtIGZlbS5ucHogWy0td2lkdGggMTZdIFstLWRldmljZSBjdWRhOjBdDQpUaGUgdXBzdHJlYW0gcmVwbyBpcyByZWFkL2ltcG9ydGVkLCBuZXZlciBtb2RpZmllZC4gSXRzIGxpY2VuY2UgZmlsZSBpcyBtaXNzaW5nIChweXByb2plY3QgZGVjbGFyZXMgTUlUKTogdGhpcyB1c2VzIGl0IGFzIGEgZG9jdW1lbnRlZCBleHRlcm5hbCByZWZlcmVuY2UuDQoiIiINCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQganNvbg0KaW1wb3J0IG1hdGgNCmltcG9ydCBzeXMNCmltcG9ydCB0aW1lDQppbXBvcnQgdHlwZXMNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KDQphcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkNCmFwLmFkZF9hcmd1bWVudCgiLS1kYW5pZWwtcm9vdCIsIHJlcXVpcmVkPVRydWUpDQphcC5hZGRfYXJndW1lbnQoIi0tYXJtIiwgY2hvaWNlcz0oIm1scCIsICJyYmYiKSwgZGVmYXVsdD0ibWxwIikNCmFwLmFkZF9hcmd1bWVudCgiLS13aWR0aCIsIHR5cGU9aW50LCBkZWZhdWx0PTE2KQ0KYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0yMDI2MDcyOCkNCmFwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD0yMDAwMCkNCmFwLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PSJjcHUiKQ0KYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIHJlcXVpcmVkPVRydWUpDQphcC5hZGRfYXJndW1lbnQoIi0tZmVtIiwgcmVxdWlyZWQ9VHJ1ZSkNCmFwLmFkZF9hcmd1bWVudCgiLS1pbnRlcmlvciIsIHR5cGU9aW50LCBkZWZhdWx0PTIwNDgpDQphcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTFlLTMpDQphcC5hZGRfYXJndW1lbnQoIi0tc25hcHNob3QtZXZlcnkiLCB0eXBlPWludCwgZGVmYXVsdD0yMDAwKQ0KYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQ0KDQpyb290ID0gUGF0aChhcmdzLmRhbmllbF9yb290KS5yZXNvbHZlKCkNCnN5cy5wYXRoWzowXSA9IFtzdHIocm9vdCAvICJzcmMiKSwgc3RyKHJvb3QpXQ0Kc3R1YiA9IHR5cGVzLk1vZHVsZVR5cGUoInRvcmNoc3VtbWFyeSIpDQpzdHViLnN1bW1hcnkgPSBsYW1iZGEgKmEsICoqazogTm9uZQ0Kc3lzLm1vZHVsZXMuc2V0ZGVmYXVsdCgidG9yY2hzdW1tYXJ5Iiwgc3R1YikNCg0KaW1wb3J0IG51bXB5IGFzIG5wICAjIG5vcWE6IEU0MDINCmltcG9ydCB0b3JjaCAgIyBub3FhOiBFNDAyDQpmcm9tIHRvcmNoIGltcG9ydCBubiAgIyBub3FhOiBFNDAyDQoNCmltcG9ydCBwaW5uX3BpZXpvLmNvbmZpZyBhcyBwY2ZnICAjIG5vcWE6IEU0MDINCmZyb20gcGlubl9waWV6by5pbmRpcmVjdCBpbXBvcnQgbW9kZWwgYXMgbW9kZWxfbW9kICAjIG5vcWE6IEU0MDINCg0Kb3V0ID0gUGF0aChhcmdzLm91dCkNCm91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQpwY2ZnLlJVTlNfRElSID0gb3V0IC8gInJ1bnMiDQpwY2ZnLlJVTlNfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCg0KeiA9IG5wLmxvYWQoYXJncy5mZW0pDQpGRU0gPSB7azogeltrXSBmb3IgayBpbiAoInh5IiwgInUiLCAidiIsICJwaGkiKX0NCkNBUFRVUkUgPSB7fQ0KDQoNCmNsYXNzIFJCRk5ldChubi5Nb2R1bGUpOg0KICAgICIiIkRyb3AtaW4gZm9yIGBtb2RlbC5uZXRgIG9mIHRoZSBwaGFzZS1lbnJpY2hlZCB0cnVuazogaW5wdXRzICh4aSwgZXRhLCB8ZXRhLS41fCwgcGhhc2UpIC0+IDggcmF3IG91dHB1dHMuIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgd2lkdGgpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgZnJvbSBtYXRoIGltcG9ydCBzcXJ0ICAjIG5vcWE6IEY0MDENCg0KICAgICAgICBjbGFzcyBFZGdlKG5uLk1vZHVsZSk6DQogICAgICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgaSwgbywgYmFzZXM9MTEpOg0KICAgICAgICAgICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICAgICAgICAgIHNwID0gMi4wIC8gKGJhc2VzIC0gMSkNCiAgICAgICAgICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiY2VudGVycyIsIHRvcmNoLmxpbnNwYWNlKC0xLjAsIDEuMCwgYmFzZXMpKQ0KICAgICAgICAgICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJ3aWR0aCIsIHRvcmNoLnRlbnNvcigxLjc1ICogc3ApKQ0KICAgICAgICAgICAgICAgIHNlbGYuY29lZmZpY2llbnRzID0gbm4uUGFyYW1ldGVyKHRvcmNoLnJhbmRuKG8sIGksIGJhc2VzKSAqICgwLjEgLyBtYXRoLnNxcnQoaSkpKQ0KICAgICAgICAgICAgICAgIHNlbGYuYmFzZSA9IG5uLlBhcmFtZXRlcih0b3JjaC5yYW5kbihvLCBpKSAqICgwLjEgLyBtYXRoLnNxcnQoaSkpKQ0KDQogICAgICAgICAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgICAgICAgICBiID0gdG9yY2guZXhwKC0oKHgudW5zcXVlZXplKC0xKSAtIHNlbGYuY2VudGVycykgLyBzZWxmLndpZHRoKS5zcXVhcmUoKSkNCiAgICAgICAgICAgICAgICByZXR1cm4gdG9yY2guZWluc3VtKCJiaWcsb2lnLT5ibyIsIGIsIHNlbGYuY29lZmZpY2llbnRzKSArIHRvcmNoLm5uLmZ1bmN0aW9uYWwubGluZWFyKHRvcmNoLm5uLmZ1bmN0aW9uYWwuc2lsdSh4KSwgc2VsZi5iYXNlKQ0KDQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uTW9kdWxlTGlzdChbRWRnZSg0LCB3aWR0aCksIEVkZ2Uod2lkdGgsIHdpZHRoKSwgRWRnZSh3aWR0aCwgOCldKQ0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgeik6DQogICAgICAgIGggPSB0b3JjaC5jYXQoWzIgKiB6WzosIDA6MV0gLSAxLCAyICogels6LCAxOjJdIC0gMSwgNCAqIHpbOiwgMjozXSAtIDEsIHpbOiwgMzo0XV0sIGRpbT0xKQ0KICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnNbOi0xXToNCiAgICAgICAgICAgIGggPSB0b3JjaC50YW5oKGxheWVyKGgpKQ0KICAgICAgICByZXR1cm4gc2VsZi5sYXllcnNbLTFdKGgpDQoNCg0KX29yaWdfYnVpbGQgPSBtb2RlbF9tb2QuYnVpbGRfZGVmYXVsdF9tb2RlbA0KDQoNCmRlZiBidWlsZF9wYXRjaGVkKCphLCAqKmspOg0KICAgIG0gPSBfb3JpZ19idWlsZCgqYSwgKiprKQ0KICAgIGlmIGFyZ3MuYXJtID09ICJyYmYiOg0KICAgICAgICBkZXYgPSBuZXh0KG0ucGFyYW1ldGVycygpKS5kZXZpY2UNCiAgICAgICAgbS5uZXQgPSBSQkZOZXQoYXJncy53aWR0aCkudG8oZGV2aWNlPWRldiwgZHR5cGU9bmV4dChtLnBhcmFtZXRlcnMoKSkuZHR5cGUpDQogICAgQ0FQVFVSRVsibW9kZWwiXSA9IG0NCiAgICByZXR1cm4gbQ0KDQoNCm1vZGVsX21vZC5idWlsZF9kZWZhdWx0X21vZGVsID0gYnVpbGRfcGF0Y2hlZA0KDQoNCmRlZiBzY29yZShtLCBkZXYpOg0KICAgIHh5ID0gdG9yY2gudGVuc29yKEZFTVsieHkiXSwgZHR5cGU9dG9yY2guZmxvYXQ2NCwgZGV2aWNlPWRldikNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgbyA9IG0oeHkpLmRvdWJsZSgpLmNwdSgpLm51bXB5KCkNCiAgICAgICAgdGlwID0gZmxvYXQobSh0b3JjaC50ZW5zb3IoW1swLjEsIDAuMF1dLCBkdHlwZT10b3JjaC5mbG9hdDY0LCBkZXZpY2U9ZGV2KSlbMCwgMV0pDQoNCiAgICBkZWYgcmVsKGEsIGIpOg0KICAgICAgICByZXR1cm4gZmxvYXQobnAubGluYWxnLm5vcm0oYSAtIGIpIC8gbnAubGluYWxnLm5vcm0oYikpDQoNCiAgICByZXR1cm4geyJ1X3JlbEwyIjogcmVsKG9bOiwgMF0sIEZFTVsidSJdKSwgInZfcmVsTDIiOiByZWwob1s6LCAxXSwgRkVNWyJ2Il0pLCAicGhpX3JlbEwyIjogcmVsKG9bOiwgMl0sIEZFTVsicGhpIl0pLA0KICAgICAgICAgICAgInRpcF92IjogdGlwLCAidGlwX3ZfZmVtIjogZmxvYXQoRkVNWyJ2Il1bbnAuYXJnbWluKG5wLmFicyhGRU1bInh5Il1bOiwgMF0gLSAwLjEpICsgRkVNWyJ4eSJdWzosIDFdKV0pfQ0KDQoNCiMgLS0tIHNuYXBzaG90IGhvb2s6IHdyYXAgdGhlIHVwc3RyZWFtIHRyYWluKCkgc28gd2UgY2FuIHNjb3JlIHBlcmlvZGljYWxseSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCmltcG9ydCBwaW5uX3BpZXpvLmluZGlyZWN0LnRyYWluIGFzIHRyYWluX21vZCAgIyBub3FhOiBFNDAyDQoNClNOQVBTID0gW10NClQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KX29yaWdfcnVuX2FkYW0gPSB0cmFpbl9tb2QucnVuX2FkYW0NCg0KDQpkZWYgcnVuX2FkYW1fcGF0Y2hlZChtb2RlbCwgdGVuc29ycywgKmEsICoqayk6DQogICAgZGV2ID0gbmV4dChtb2RlbC5wYXJhbWV0ZXJzKCkpLmRldmljZQ0KICAgIGVwX3RvdGFsID0gay5nZXQoImVwb2NocyIsIGFyZ3MuZXBvY2hzKQ0KICAgIGNodW5rID0gYXJncy5zbmFwc2hvdF9ldmVyeQ0KICAgIGRvbmUgPSAwDQogICAgcmVzdWx0ID0gTm9uZQ0KICAgIHdoaWxlIGRvbmUgPCBlcF90b3RhbDoNCiAgICAgICAgbiA9IG1pbihjaHVuaywgZXBfdG90YWwgLSBkb25lKQ0KICAgICAgICBrayA9IGRpY3QoaykNCiAgICAgICAga2tbImVwb2NocyJdID0gbg0KICAgICAgICByZXMgPSBfb3JpZ19ydW5fYWRhbShtb2RlbCwgdGVuc29ycywgKmEsICoqa2spIGlmIHJlc3VsdCBpcyBOb25lIGVsc2UgX29yaWdfcnVuX2FkYW0obW9kZWwsIHRlbnNvcnMsICphLCAqKnsqKmtrLCAibG9zc193ZWlnaHRzIjogcmVzdWx0WzFdfSkNCiAgICAgICAgcmVzdWx0ID0gcmVzIGlmIHJlc3VsdCBpcyBOb25lIGVsc2UgKHJlc3VsdFswXSArIHJlc1swXSwgcmVzWzFdLCBtaW4ocmVzdWx0WzJdLCByZXNbMl0pKQ0KICAgICAgICBkb25lICs9IG4NCiAgICAgICAgc2MgPSBzY29yZShtb2RlbCwgZGV2KQ0KICAgICAgICBTTkFQUy5hcHBlbmQoeyJzdGVwIjogZG9uZSwgInUiOiBzY1sidV9yZWxMMiJdLCAidiI6IHNjWyJ2X3JlbEwyIl0sICJwaGkiOiBzY1sicGhpX3JlbEwyIl0sICJ0aXBfdiI6IHNjWyJ0aXBfdiJdLCAidCI6IHRpbWUucGVyZl9jb3VudGVyKCkgLSBUMH0pDQogICAgICAgIHByaW50KGpzb24uZHVtcHMoU05BUFNbLTFdKSwgZmx1c2g9VHJ1ZSkNCiAgICByZXR1cm4gcmVzdWx0DQoNCg0KdHJhaW5fbW9kLnJ1bl9hZGFtID0gcnVuX2FkYW1fcGF0Y2hlZA0KDQppbXBvcnQgc2NyaXB0cy50cmFpbl9pbmRpcmVjdCBhcyB0aSAgIyBub3FhOiBFNDAyDQoNCmFyZ3YgPSBbInRyYWluX2luZGlyZWN0IiwgIi0tZXBvY2hzLWFkYW0iLCBzdHIoYXJncy5lcG9jaHMpLCAiLS1lcG9jaHMtbGJmZ3MiLCAiMCIsICItLWxyLWFkYW0iLCBzdHIoYXJncy5sciksICItLW1vZGVsLXR5cGUiLCAicHlyYW1pZCIsDQogICAgICAgICItLWhpZGRlbi1zaXplcyIsICI1MCIsICI1MCIsICI1MCIsICItLWFjdGl2YXRpb24iLCAic2lsdSIsICItLXBoYXNlLWVucmljaGVkIiwgIi0tbm8tc3BsaXQtdHJ1bmtzIiwgIi0tbm8taGFyZC1uYXR1cmFsLWJjcyIsDQogICAgICAgICItLWludGVyaW9yLXBlci1sYXllciIsIHN0cihhcmdzLmludGVyaW9yKSwgIi0tYm91bmRhcnktcG9pbnRzIiwgIjI1NiIsICItLWludGVyZmFjZS1wb2ludHMiLCAiMjU2IiwgIi0tcmVzYW1wbGUtZXZlcnkiLCAiMjAwIiwNCiAgICAgICAgIi0taW50ZXJmYWNlLXdlaWdodCIsICIxLjAiLCAiLS1uby1zbGVuZGVyLXRyYW5zdmVyc2Utc2NhbGluZyIsICItLW5vcm1hbC1jb25zdGl0dXRpdmUtd2VpZ2h0IiwgIjEuMCIsDQogICAgICAgICItLWNvbnN0aXR1dGl2ZS1ub3JtYWxpemF0aW9uIiwgInJlcHJlc2VudGVkIiwgIi0tbm8tY29uc3RpdHV0aXZlLXRyYWN0aW9uLWJjcyIsICItLW5vLWNvbnN0aXR1dGl2ZS1mbHV4LXN0b3BncmFkIiwNCiAgICAgICAgIi0tbWVjaGFuaWNhbC1jb25zdGl0dXRpdmUtc3RvcGdyYWQiLCAiLS1zdHJlc3Mtd2FybXVwLWVwb2NocyIsICIwIiwgIi0tc2VlZCIsIHN0cihhcmdzLnNlZWQpLCAiLS12b2x0YWdlIiwgIjEwMCIsICItLW5vLXplcm8tb3V0cHV0IiwNCiAgICAgICAgIi0tbm8tcGVyaW9kaWMtY2hlY2twb2ludHMiLCAiLS1kdHlwZSIsICJmbG9hdDY0IiwgIi0tZGV2aWNlIiwgKCJjdWRhIiBpZiBhcmdzLmRldmljZS5zdGFydHN3aXRoKCJjdWRhIikgZWxzZSBhcmdzLmRldmljZSksICItLXJ1bi1uYW1lIiwgZiJkYW5pZWwte2FyZ3MuYXJtfS1zZWVkLXthcmdzLnNlZWR9Il0NCnN5cy5hcmd2ID0gYXJndg0KdGkubWFpbigpDQptID0gQ0FQVFVSRVsibW9kZWwiXQ0KZGV2ID0gbmV4dChtLnBhcmFtZXRlcnMoKSkuZGV2aWNlDQpyZXMgPSB7InN0YXR1cyI6ICJEUkFGVF9VTlZFUklGSUVEIiwgImFybSI6IGFyZ3MuYXJtLCAid2lkdGgiOiBhcmdzLndpZHRoIGlmIGFyZ3MuYXJtID09ICJyYmYiIGVsc2UgTm9uZSwgInNlZWQiOiBhcmdzLnNlZWQsICJlcG9jaHMiOiBhcmdzLmVwb2NocywNCiAgICAgICAicGFyYW1zIjogc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtLnBhcmFtZXRlcnMoKSksICJzZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIFQwLCAic2NvcmVfZmluYWwiOiBzY29yZShtLCBkZXYpLCAic25hcHNob3RzIjogU05BUFMsDQogICAgICAgInVwc3RyZWFtX2NvbW1pdCI6ICJlNzg1NmYxICgyMDI2LTEwLTAyKSIsICJzY2llbnRpZmljX2FjY2VwdGVkIjogRmFsc2UsICJyZWZlcmVuY2VfYWNjZXB0ZWQiOiBGYWxzZX0NCnRvcmNoLnNhdmUoeyJzdGF0ZV9kaWN0IjogbS5zdGF0ZV9kaWN0KCl9LCBvdXQgLyBmImNoZWNrcG9pbnQte2FyZ3MuYXJtfS1zZWVkLXthcmdzLnNlZWR9LnB0IikNCihvdXQgLyAicnVuLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzLCBpbmRlbnQ9MikpDQpwcmludChqc29uLmR1bXBzKHsiZmluYWwiOiByZXNbInNjb3JlX2ZpbmFsIl0sICJwYXJhbXMiOiByZXNbInBhcmFtcyJdLCAicyI6IHJlc1sic2Vjb25kcyJdfSksIGZsdXNoPVRydWUpDQo=', 'fem.npz': '...'}
for k, v in BLOB.items(): (p / k).write_bytes(base64.b64decode(v))
zipfile.ZipFile(p / "upstream.zip").extractall(p / "daniel")
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(json.dumps({"gpus": names, "torch": torch.__version__, "hashes": {'upstream.zip': '4a7bf5ae9097e5009091ee459ef6c07f18ded9070da0c0c38df51f3a6ee51d32', 'daniel_arm.py': '937aa693bd0dc30763a43683148358d9756d71f276c155eb4ded294ab4ac14cc', 'fem.npz': '39bc04a5ec51f71d229c96ffb011c5d32d96e1411a9733bbbc99178bda1e36e6'}}), flush=True)
assert names and all("T4" in x for x in names), names
JOBS = [{'tag': 'rbf16-seed-20260729', 'arm': 'rbf', 'seed': 20260729, 'epochs': 20000, 'width': 16}, {'tag': 'rbf16-seed-20260730', 'arm': 'rbf', 'seed': 20260730, 'epochs': 20000, 'width': 16}, {'tag': 'mlp-seed-20260729', 'arm': 'mlp', 'seed': 20260729, 'epochs': 20000}, {'tag': 'mlp-seed-20260730', 'arm': 'mlp', 'seed': 20260730, 'epochs': 20000}]
running, ledger, queue = {}, [], list(JOBS)
while queue or running:
    for g in list(running):
        pr, tag, st = running[g]
        if pr.poll() is not None: ledger.append({"job": tag, "rc": pr.returncode, "wall": time.monotonic() - st}); del running[g]
    if time.monotonic() - T0 > CAP:
        for g, (pr, tag, st) in running.items(): pr.kill()
        ledger.append({"note": "cap reached, killed", "running": [t for _, t, _ in running.values()]}); break
    for g in range(len(names)):
        if g not in running and queue:
            j = queue.pop(0); tag = j["tag"]
            cmd = [sys.executable, str(p / "daniel_arm.py"), "--daniel-root", str(p / "daniel"), "--arm", j["arm"], "--seed", str(j["seed"]),
                   "--epochs", str(j["epochs"]), "--width", str(j.get("width", 16)), "--device", "cuda", "--fem", str(p / "fem.npz"), "--out", str(w / "results" / tag)]
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
            running[g] = (subprocess.Popen(cmd, stdout=open(w / f"{tag}.log", "w"), stderr=subprocess.STDOUT, env=env), tag, time.monotonic())
    time.sleep(3)
summ = {"wall": time.monotonic() - T0, "cap": CAP, "ledger": ledger, "gpus": names, "scientific_accepted": False, "reference_accepted": False}
(w / "session_summary.json").write_text(json.dumps(summ, indent=1)); print(json.dumps(summ))
